# 03 - Split comparison

Compare the three run configurations:

| Label | Split | Hour feature | Files |
|---|---|---|---|
| random | stratified random 80/20 | yes | `model_random.joblib`, `metrics_random.json` |
| temporal | last 20% of the time range is test | yes | `model_temporal.joblib`, `metrics_temporal.json` |
| temporal_nohour | same as temporal | no | `model_temporal_nohour.joblib`, `metrics_temporal_nohour.json` |

Each configuration is produced with `python -m src.train` and `python -m src.evaluate`
(with `--split` / `--no-hour`). Configurations that were not run yet are skipped.

The metrics files have point estimates only. To get confidence intervals and PR curves,
this notebook reloads each saved model and recomputes its predictions on its own test set.
The test sets are only used for evaluation here; nothing is tuned on them.

## Setup

In [ ]:
import json
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from sklearn.metrics import average_precision_score, precision_recall_curve

from src.config import SEED, metrics_path, model_path
from src.data import load_processed, split_data
from src.evaluate import metrics_at
from src.features import add_features

N_BOOT = 1000  # bootstrap resamples for the confidence intervals

CONFIGS = {
    "random": ("random", True),
    "temporal": ("temporal", True),
    "temporal_nohour": ("temporal", False),
}

## Load metrics, models and test predictions

For each configuration, load the metrics file and the model, redo the same train/test
split, and predict on the test set. As a sanity check, the recomputed PR-AUC must match
the one in the metrics file.

In [ ]:
df = load_processed()
runs = {}

for label, (split, use_hour) in CONFIGS.items():
    m_path, b_path = metrics_path(split, use_hour), model_path(split, use_hour)
    if not (m_path.exists() and b_path.exists()):
        print(f"{label}: skipped (missing {m_path.name} or {b_path.name})")
        continue

    metrics = json.loads(m_path.read_text())
    bundle = joblib.load(b_path)
    X_train, X_test, y_train, y_test = split_data(df, method=split)
    proba = bundle["pipeline"].predict_proba(X_test[bundle["input_columns"]])[:, 1]

    assert np.isclose(average_precision_score(y_test, proba), metrics["pr_auc"]), label
    runs[label] = {
        "metrics": metrics,
        "bundle": bundle,
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train.to_numpy(),
        "y_test": y_test.to_numpy(),
        "proba": proba,
    }
    print(f"{label}: loaded ({len(y_test):,} test rows, {int(y_test.sum())} frauds)")

## 1. Comparison table with bootstrap confidence intervals

95% intervals come from a percentile bootstrap: the test set is resampled with
replacement `N_BOOT` times and each metric is recomputed. Precision, recall and F1 use
the threshold chosen by cross-validation on the training set (stored in the model file).

How to read it:

- The test sets have few frauds (tens, not thousands), so the intervals are wide.
  Each missed or caught fraud moves recall by more than one point.
- **random vs temporal are evaluated on different test sets** (different rows, different
  fraud rates). Their numbers answer different questions: "how well does the model do on
  rows mixed in time with the training data" vs "how well does it do on the future".
  Comparing them only says how optimistic the random split is; overlapping intervals
  mean the data does not show a clear difference.
- **temporal vs temporal_nohour share the same test set**, so their difference can be
  measured with a paired bootstrap (same resampled rows for both models). That is done
  in the next cell.

In [ ]:
def bootstrap_metrics(y, proba, threshold, n_boot=N_BOOT, seed=SEED, idx_list=None):
    """Bootstrap PR-AUC, precision, recall and F1 at `threshold`.

    Returns a DataFrame with one row per resample. Pass idx_list to reuse the same
    resampled rows across models (paired bootstrap).
    """
    if idx_list is None:
        rng = np.random.default_rng(seed)
        idx_list = [rng.integers(0, len(y), len(y)) for _ in range(n_boot)]
    rows = []
    for idx in idx_list:
        y_b, p_b = y[idx], proba[idx]
        if y_b.sum() == 0:
            continue
        at = metrics_at(y_b, p_b, threshold)
        rows.append(
            {
                "pr_auc": average_precision_score(y_b, p_b),
                "precision": at["precision"],
                "recall": at["recall"],
                "f1": at["f1"],
            }
        )
    return pd.DataFrame(rows)


def fmt_ci(point, samples):
    low, high = np.percentile(samples, [2.5, 97.5])
    return f"{point:.3f} [{low:.3f}, {high:.3f}]"


boot = {}
table = []
for label, run in runs.items():
    m = run["metrics"]
    at = m["at_chosen_threshold"]
    boot[label] = bootstrap_metrics(run["y_test"], run["proba"], at["threshold"])
    b = boot[label]
    table.append(
        {
            "config": label,
            "PR-AUC [95% CI]": fmt_ci(m["pr_auc"], b["pr_auc"]),
            "precision [95% CI]": fmt_ci(at["precision"], b["precision"]),
            "recall [95% CI]": fmt_ci(at["recall"], b["recall"]),
            "F1 [95% CI]": fmt_ci(at["f1"], b["f1"]),
            "threshold": round(at["threshold"], 4),
            "test frauds": m["n_test_frauds"],
            "train fraud rate": f"{m['n_train_frauds'] / m['n_train']:.4%}",
            "test fraud rate": f"{m['n_test_frauds'] / m['n_test']:.4%}",
        }
    )

pd.DataFrame(table).set_index("config")

### Are the differences larger than the noise?

- **Overlap check (any two configurations):** if the 95% intervals overlap, the data
  does not show a clear difference. Overlap does not prove the models are equal, and
  for random vs temporal the test sets differ, so this is only a rough guide.
- **Paired difference (temporal − temporal_nohour):** same test rows, so the bootstrap
  gives an interval for the difference itself. If it contains 0, the effect of `Hour`
  is within the noise.

In [ ]:
def ci(samples):
    return np.percentile(samples, [2.5, 97.5])


labels = list(boot)
for i, a in enumerate(labels):
    for b_label in labels[i + 1 :]:
        for metric in ["pr_auc", "f1"]:
            lo_a, hi_a = ci(boot[a][metric])
            lo_b, hi_b = ci(boot[b_label][metric])
            overlap = lo_a <= hi_b and lo_b <= hi_a
            verdict = "intervals overlap (no clear difference)" if overlap else "intervals do not overlap"
            print(f"{a} vs {b_label}, {metric}: {verdict}")

if {"temporal", "temporal_nohour"} <= set(runs):
    y = runs["temporal"]["y_test"]
    rng = np.random.default_rng(SEED)
    idx_list = [rng.integers(0, len(y), len(y)) for _ in range(N_BOOT)]
    with_hour = bootstrap_metrics(
        y, runs["temporal"]["proba"],
        runs["temporal"]["metrics"]["at_chosen_threshold"]["threshold"], idx_list=idx_list,
    )
    no_hour = bootstrap_metrics(
        y, runs["temporal_nohour"]["proba"],
        runs["temporal_nohour"]["metrics"]["at_chosen_threshold"]["threshold"], idx_list=idx_list,
    )
    for metric in ["pr_auc", "f1"]:
        diff = with_hour[metric] - no_hour[metric]
        point = (
            runs["temporal"]["metrics"]["pr_auc"] - runs["temporal_nohour"]["metrics"]["pr_auc"]
            if metric == "pr_auc"
            else runs["temporal"]["metrics"]["at_chosen_threshold"]["f1"]
            - runs["temporal_nohour"]["metrics"]["at_chosen_threshold"]["f1"]
        )
        lo, hi = ci(diff)
        inside = "contains 0: within the noise" if lo <= 0 <= hi else "does not contain 0"
        print(f"temporal - temporal_nohour, {metric}: {point:+.3f} [{lo:+.3f}, {hi:+.3f}] -> {inside}")
else:
    print("Paired comparison skipped: needs both temporal and temporal_nohour.")

## 2. Precision-Recall curves

One panel per configuration, each on its own test set. The dashed line is the test fraud
rate, which is what a random classifier would get. Because the fraud rates differ between
test sets, the baselines differ too; compare each curve with its own baseline.

In [ ]:
fig, axes = plt.subplots(1, len(runs), figsize=(5 * len(runs), 4.5), squeeze=False)
for ax, (label, run) in zip(axes[0], runs.items()):
    y, proba = run["y_test"], run["proba"]
    precision, recall, _ = precision_recall_curve(y, proba)
    ax.plot(recall, precision, color="#d1495b", linewidth=2,
            label=f"AP = {average_precision_score(y, proba):.3f}")
    ax.axhline(y.mean(), color="#555555", linestyle="--", linewidth=1,
               label=f"fraud rate = {y.mean():.4f}")
    ax.set_title(f"{label} ({int(y.sum())} test frauds)")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.05)
    ax.legend(loc="lower left")
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

The curves are step-shaped at high recall because there are few frauds: each one that is
ranked low causes a visible drop. Small differences between panels should be read
together with the intervals in section 1, not on their own.

## 3. Drift between train and test (temporal split)

Two-sample Kolmogorov-Smirnov test (`scipy.stats.ks_2samp`) between the train and test
rows of the temporal split, for the 10 most important features of the temporal model.
The statistic **D** is the largest gap between the two cumulative distributions (0 = same
distribution, 1 = no overlap).

As a reference, the same test is run between train and test of the **random** split.
Those sets are drawn from the same distribution by construction, so their D shows how
large D gets from sampling noise alone.

Notes:

- With ~227k train and ~57k test rows, the p-value is tiny even for very small shifts.
  Look at D, and at D compared with the random-split reference, rather than at p.
- The test on all rows mostly reflects legitimate transactions (>99.8%). The
  "frauds only" columns repeat it on the fraud rows alone; with only 74 test frauds that
  test has much less power, so there the p-value is informative.
- `Hour` drifts by construction: the temporal test set covers only the last hours of
  the second day, so it contains only part of the 24 hours.

In [ ]:
if "temporal" not in runs:
    print("Skipped: needs the temporal model.")
else:
    temporal = runs["temporal"]
    classifier = temporal["bundle"]["pipeline"].named_steps["classifier"]
    importances = pd.Series(
        classifier.feature_importances_, index=classifier.feature_names_in_
    ).sort_values(ascending=False)
    top10 = importances.head(10).index

    use_hour = temporal["bundle"]["use_hour"]
    feats_train_t = add_features(temporal["X_train"], use_hour=use_hour)
    feats_test_t = add_features(temporal["X_test"], use_hour=use_hour)
    X_train_r, X_test_r, _, _ = split_data(df, method="random")
    feats_train_r = add_features(X_train_r, use_hour=use_hour)
    feats_test_r = add_features(X_test_r, use_hour=use_hour)

    fraud_train = temporal["y_train"] == 1
    fraud_test = temporal["y_test"] == 1

    rows = []
    for feature in top10:
        ks_t = ks_2samp(feats_train_t[feature], feats_test_t[feature])
        ks_r = ks_2samp(feats_train_r[feature], feats_test_r[feature])
        ks_f = ks_2samp(feats_train_t[feature][fraud_train], feats_test_t[feature][fraud_test])
        rows.append(
            {
                "feature": feature,
                "importance": importances[feature],
                "D temporal": ks_t.statistic,
                "p temporal": ks_t.pvalue,
                "D random (reference)": ks_r.statistic,
                "D ratio": ks_t.statistic / ks_r.statistic,
                "D frauds only": ks_f.statistic,
                "p frauds only": ks_f.pvalue,
            }
        )
    drift = pd.DataFrame(rows).set_index("feature")
    display(drift.style.format({"importance": "{:.3f}", "D temporal": "{:.3f}",
                                "p temporal": "{:.1e}", "D random (reference)": "{:.3f}",
                                "D ratio": "{:.1f}", "D frauds only": "{:.3f}",
                                "p frauds only": "{:.3f}"}))

    fig, ax = plt.subplots(figsize=(7, 4))
    positions = np.arange(len(drift))
    ax.barh(positions + 0.2, drift["D temporal"], height=0.4, color="#d1495b", label="temporal split")
    ax.barh(positions - 0.2, drift["D random (reference)"], height=0.4, color="#999999",
            label="random split (reference)")
    ax.set_yticks(positions, drift.index)
    ax.invert_yaxis()
    ax.set_xlabel("KS statistic D (train vs test)")
    ax.set_title("Train/test drift for the top 10 features")
    ax.legend()
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    plt.show()

**Reading of the current run** (`model_temporal.joblib`, trained on 2026-11-13; re-check
if the models are retrained):

- All 10 features differ more between train and test in the temporal split than in the
  random split: D is roughly 6 to 95 times the random-split reference. So there is a
  measurable distribution shift, not just sampling noise.
- Most shifts are moderate (D between about 0.03 and 0.15). The largest is `V3`
  (D ≈ 0.35), followed by `V11` and `V4` (D ≈ 0.13-0.14).
- On fraud rows only, most of the top features also differ (D ≈ 0.23-0.40, p < 0.01),
  while `V14` and `V4` do not show a significant shift. With 74 test frauds and 10 tests,
  this is suggestive, not precise.
- The temporal test set covers only hours 16-23 (relative to the first transaction) of
  the second day. Part of the shift may be a time-of-day pattern rather than a change
  over days; with only two days of data these cannot be separated.
- This section shows that the distributions differ. It does **not** measure how much
  the drift hurts the model; section 1 is the place for that, and with a single
  temporal run there is nothing to compare against yet.

## 4. Threshold trade-off (temporal split, test set)

Precision, recall and false alarms per 10,000 transactions (all test transactions, not
only legitimate ones) for thresholds from 0.1 to 0.9.

**This table is descriptive only.** It is computed on the test set, so it must not be used
to pick the threshold; doing so would make the test metrics optimistic. The threshold in
the saved model was chosen by cross-validation on the training set.

In [ ]:
if "temporal" not in runs:
    print("Skipped: needs the temporal model.")
else:
    y, proba = runs["temporal"]["y_test"], runs["temporal"]["proba"]
    rows = []
    for threshold in np.round(np.arange(0.1, 1.0, 0.1), 1):
        at = metrics_at(y, proba, threshold)
        rows.append(
            {
                "threshold": threshold,
                "precision": at["precision"],
                "recall": at["recall"],
                "frauds caught": f"{at['tp']} / {int(y.sum())}",
                "false alarms": at["fp"],
                "false alarms per 10k": at["fp"] / len(y) * 10_000,
            }
        )
    tradeoff = pd.DataFrame(rows).set_index("threshold")
    print(f"Threshold chosen by CV on train: {runs['temporal']['metrics']['at_chosen_threshold']['threshold']:.4f}")
    display(tradeoff.style.format({"precision": "{:.3f}", "recall": "{:.3f}",
                                   "false alarms per 10k": "{:.2f}"}))

**Reading of the current run:**

- Raising the threshold from 0.1 to 0.9 lowers recall from about 0.81 to 0.46 and false
  alarms from about 10 to 0 per 10,000 transactions.
- Between 0.3 and 0.5, precision is 0.84-0.93 and recall 0.70-0.78, with 1-2 false alarms
  per 10,000 transactions. The CV threshold (≈ 0.36) falls in this range.
- Neighbouring rows often differ by only 1-5 caught frauds, which is within the noise
  shown in section 1. Small differences between thresholds should not be over-read.

## 5. Feature importance (temporal model)

Impurity-based importance (mean decrease in impurity) of the Random Forest in the
temporal model, computed on the training data.

Caveats:

- It is computed on the training set and says which features the trees used to split,
  not how much each feature helps on new data. Permutation importance on held-out data
  would answer that better.
- It is biased towards continuous features with many distinct values, and correlated
  features share importance between them.
- `V1`-`V28` are anonymized PCA components, so a high importance does not tell us which
  real-world attribute matters.

In [ ]:
if "temporal" not in runs:
    print("Skipped: needs the temporal model.")
else:
    top20 = importances.head(20)
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.barh(top20.index[::-1], top20.values[::-1], color="#d1495b")
    ax.set_xlabel("Mean decrease in impurity")
    ax.set_title("Top 20 features, temporal model")
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    plt.show()

    print(f"Top 5 share of total importance: {importances.head(5).sum():.1%}")
    if "Hour" in importances:
        print(f"Hour: rank {importances.index.get_loc('Hour') + 1} of {len(importances)}, "
              f"importance {importances['Hour']:.3f}")

**Reading of the current run:**

- `V14`, `V10`, `V4`, `V17` and `V12` are the top 5 and together hold about 55% of the
  total importance.
- `Hour` is near the bottom (rank 29 of 30, importance ≈ 0.005): the trees rarely split
  on it. This suggests that removing it would change little, but impurity importance
  cannot show that; the `temporal_nohour` run is the direct test.
- `V3`, the feature with the largest drift in section 3, is 6th in importance.

## Summary

What the current numbers support (only the `temporal` configuration has been run):

- On the most recent 20% of the data, the temporal model reaches PR-AUC 0.817 with a
  95% bootstrap interval of about [0.74, 0.90]. The interval is wide because the test set
  has only 74 frauds.
- Train and test of the temporal split have clearly different feature distributions.
- `Hour` has very low impurity importance in the temporal model.

What they do **not** support yet:

- Whether the random split gives optimistic results: `random` has not been run with the
  current code.
- Whether `Hour` helps or hurts: `temporal_nohour` has not been run.
- Whether the drift lowers performance, or by how much.

After running the missing configurations, re-run this notebook and update the text in
sections 3-5 and this summary to match the new numbers.